# SmartSentry AML — Pipeline Orchestrator

**One-click sequential execution of the entire AML pipeline.**

This notebook runs all 6 modules in order, validates outputs between stages, and produces a final summary report. Each module is executed as a subprocess using `nbconvert`, so kernel state is isolated and any single-module failure is caught without crashing the entire pipeline.

### Pipeline Order
```
[00] aml_generator_complete_pipeline.ipynb   → Synthetic data generation
[01] 01__aml_typology_detector.ipynb         → Graph-based typology detection
[02] 02__aml_rules_engine.ipynb              → 126-rule compliance engine
[03] 03__aml_feature_engineering.ipynb        → Velocity/balance/IP features
[04] 04__aml_ml_preparation.ipynb            → Phase 1: Binary AML detection
[05] 05__aml_phase2_typology_classifier.ipynb → Phase 2: Typology classification
```


## 0 — Pipeline Mode Selection

**Choose pipeline mode:**
- **`yes`** — Run the full training pipeline (generate synthetic transactions → detector → rules → features → train Phase 1 → train Phase 2). Use this when you want to (re)train models.
- **`no`** — Run the **inference orchestrator** against an existing user-provided transaction file. The saved model bundles will be loaded and applied. Use this when you want to score new data.

The cell below asks for `yes` or `no` and dispatches accordingly.

In [2]:
# ─── Pipeline mode prompt ─────────────────────────────────
# The user is asked whether to run the FULL training pipeline (`yes`) or
# the INFERENCE-ONLY orchestrator (`no`).
#
# Override programmatically by setting environment variable AML_PIPELINE_MODE
# to "train" or "predict".
import os, sys, subprocess

_env_mode = os.environ.get("AML_PIPELINE_MODE", "").strip().lower()
if _env_mode in ("train", "predict"):
    _user_choice = "yes" if _env_mode == "train" else "no"
    print(f"AML_PIPELINE_MODE={_env_mode!r} → answer = {_user_choice!r}")
else:
    try:
        _user_choice = input("Run full training pipeline? (yes/no): ").strip().lower()
    except EOFError:
        # When running in non-interactive contexts (e.g. nbconvert), default to train
        _user_choice = "yes"
        print(f"Non-interactive context — defaulting to 'yes' (training pipeline)")

if _user_choice in ("yes", "y", "true", "1"):
    PIPELINE_MODE = "train"
    print("\n→ Mode: TRAINING")
    print("  Full pipeline: data generation → detector → rules → features → Phase 1 → Phase 2\n")
elif _user_choice in ("no", "n", "false", "0"):
    PIPELINE_MODE = "predict"
    print("\n→ Mode: INFERENCE")
    print("  Will execute 00b__aml_inference_orchestrator.ipynb instead.")
    print("  This loads saved Phase 1 + Phase 2 bundles and scores user-provided data.\n")

    inf_path = os.path.join(os.getcwd(), "00b__aml_inference_orchestrator.ipynb")
    if not os.path.exists(inf_path):
        raise FileNotFoundError(
            f"Inference orchestrator not found at {inf_path}.\n"
            f"Ensure 00b__aml_inference_orchestrator.ipynb is in the same directory."
        )

    # Execute the inference orchestrator end-to-end via nbconvert
    out_dir = os.path.join(os.path.dirname(os.getcwd()), "outputs_updated", "executed_notebooks")
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "00b__aml_inference_orchestrator.ipynb")

    print(f"Launching {os.path.basename(inf_path)}...")
    res = subprocess.run(
        [sys.executable, "-m", "jupyter", "nbconvert",
         "--to", "notebook", "--execute",
         "--ExecutePreprocessor.timeout=14400",
         "--ExecutePreprocessor.kernel_name=python3",
         "--output", out_path, inf_path],
        capture_output=True, text=True,
    )
    if res.returncode != 0:
        print("Inference orchestrator FAILED:")
        print((res.stderr or "")[-2000:])
        raise RuntimeError("Inference pipeline failed — see stderr above")
    print("Inference orchestrator completed successfully.")
    print(f"Executed notebook saved to: {out_path}")

    # Stop further training-pipeline cells from running
    print("\n" + "=" * 70)
    print("DONE (inference mode) — skip the rest of this notebook.")
    print("=" * 70)
    # Raise a controlled exception to halt remaining cells when in nbconvert
    class _PipelineDone(Exception): pass
    raise _PipelineDone("Inference orchestrator finished — halting training pipeline.")
else:
    raise ValueError(f"Unexpected answer {_user_choice!r}. Reply 'yes' or 'no'.")



→ Mode: INFERENCE
  Will execute 00b__aml_inference_orchestrator.ipynb instead.
  This loads saved Phase 1 + Phase 2 bundles and scores user-provided data.

Launching 00b__aml_inference_orchestrator.ipynb...
Inference orchestrator completed successfully.
Executed notebook saved to: c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\executed_notebooks\00b__aml_inference_orchestrator.ipynb

DONE (inference mode) — skip the rest of this notebook.


_PipelineDone: Inference orchestrator finished — halting training pipeline.

## 1 — Setup & Configuration


In [ ]:
import os
import sys
import time
import json
import subprocess
from datetime import datetime, timedelta
from pathlib import Path

# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — Update these paths to match your environment
# ═══════════════════════════════════════════════════════════════

# Directory containing all notebooks
NOTEBOOK_DIR = os.getcwd()

# Output directory (parent-level outputs_updated)
OUTPUT_DIR = os.path.join(os.path.dirname(NOTEBOOK_DIR), "outputs_updated")

# Pipeline notebooks in execution order
PIPELINE = [
    {
        "id": "00",
        "name": "Data Generator",
        "notebook": "aml_generator_complete_pipeline.ipynb",
        "outputs": ["transactions_generated_typology.parquet"],
        "description": "Generate synthetic transactions with 10 AML typologies",
    },
    {
        "id": "01",
        "name": "Typology Detector",
        "notebook": "01__aml_typology_detector.ipynb",
        "outputs": ["stg_transactions_flagged.parquet"],
        "description": "Graph-based detection of AML patterns, assign is_aml labels",
    },
    {
        "id": "02",
        "name": "Rules Engine",
        "notebook": "02__aml_rules_engine.ipynb",
        "outputs": ["stg_transactions_rules.parquet"],
        "description": "Apply 126 regulatory compliance rules (RBI/PMLA/FIU-IND)",
    },
    {
        "id": "03",
        "name": "Feature Engineering",
        "notebook": "03__aml_feature_engineering.ipynb",
        "outputs": ["stg_transactions_features.parquet"],
        "description": "Compute velocity, balance, IP risk, and volume features",
    },
     {
        "id": "04",
        "name": "Phase 1: AML Detection",
        "notebook": "04__aml_ml_preparation.ipynb",
        # NOTE: notebook 04 writes to ../python_scripts/ml_outputs/, NOT under
        # outputs_updated/. The output paths below are checked verbatim against
        # the filesystem (no OUTPUT_DIR prefix is added when they're absolute).
        "outputs": [
            os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "ml_outputs", "final_lgb_model.txt"),
            os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "ml_outputs", "phase1_model_bundle.joblib"),
            os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "ml_outputs", "model_metadata.json"),
            os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "ml_outputs", "X_train.parquet"),
            os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "ml_outputs", "X_test.parquet"),
            os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "ml_outputs", "y_train.parquet"),
            os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "ml_outputs", "y_test.parquet"),
        ],
        "description": "Train binary AML classifier with hyperparameter tuning",
    },
    {
        "id": "05",
        "name": "Phase 2: Typology Classifier",
        "notebook": "05__aml_phase2_typology_classifier.ipynb",
        # NOTE: notebook 05 writes to ../python_scripts/phase2_outputs/, NOT under
        # outputs_updated/.
        "outputs": [
            os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "phase2_outputs", "phase2_typology_model.txt"),
            os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "phase2_outputs", "phase2_model_bundle.joblib"),
            os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "phase2_outputs", "combined_aml_output.parquet"),
            os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "phase2_outputs", "model_parameters_full.json"),
        ],
        "description": "Train 10-class typology classifier with multi-label threshold",
    },
]

# Execution settings
TIMEOUT_MINUTES = 60          # Max time per notebook (increase for large datasets)
STOP_ON_FAILURE = True        # Stop pipeline if any notebook fails
SAVE_EXECUTED_NOTEBOOKS = True  # Save executed notebooks with outputs

print("=" * 70)
print("SmartSentry AML — Pipeline Orchestrator")
print("=" * 70)
print(f"  Notebook directory:  {NOTEBOOK_DIR}")
print(f"  Output directory:    {OUTPUT_DIR}")
print(f"  Timeout per module:  {TIMEOUT_MINUTES} minutes")
print(f"  Stop on failure:     {STOP_ON_FAILURE}")
print(f"  Modules to run:      {len(PIPELINE)}")
print()

# Verify all notebooks exist
all_found = True
for stage in PIPELINE:
    nb_path = os.path.join(NOTEBOOK_DIR, stage["notebook"])
    exists = os.path.exists(nb_path)
    status = "✓" if exists else "⚠ NOT FOUND"
    print(f"  [{stage['id']}] {stage['notebook']:<55s} {status}")
    if not exists:
        all_found = False

if not all_found:
    raise FileNotFoundError("One or more notebooks are missing. Fix paths above.")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n  ✓ All notebooks found. Ready to execute.")


SmartSentry AML — Pipeline Orchestrator
  Notebook directory:  c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts
  Output directory:    c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated
  Timeout per module:  60 minutes
  Stop on failure:     True
  Modules to run:      6

  [00] aml_generator_complete_pipeline.ipynb                   ✓
  [01] 01__aml_typology_detector.ipynb                         ✓
  [02] 02__aml_rules_engine.ipynb                              ✓
  [03] 03__aml_feature_engineering.ipynb                       ✓
  [04] 04__aml_ml_preparation.ipynb                            ✓
  [05] 05__aml_phase2_typology_classifier.ipynb                ✓

  ✓ All notebooks found. Ready to execute.


In [ ]:
# Add this in Cell 2 of the orchestrator, right after OUTPUT_DIR is defined:
os.makedirs(OUTPUT_DIR, exist_ok=True)
for subdir in ["executed_notebooks"]:
    os.makedirs(os.path.join(OUTPUT_DIR, subdir), exist_ok=True)

## 2 — Notebook Runner Engine


In [ ]:
def run_notebook(notebook_path, timeout_minutes=60, working_dir=None):
    """
    Execute a Jupyter notebook using nbconvert and return status.
    The notebook runs in its own kernel (isolated state).
    """
    start_time = time.time()
    notebook_name = os.path.basename(notebook_path)
    
    # Output path for executed notebook (with cell outputs preserved)
    executed_dir = os.path.join(OUTPUT_DIR, "executed_notebooks")
    os.makedirs(executed_dir, exist_ok=True)
    executed_path = os.path.join(executed_dir, notebook_name)
    
    print(f"  Executing: {notebook_name}")
    print(f"  Working dir: {working_dir or os.path.dirname(notebook_path)}")
    
    try:
        cmd = [
            sys.executable, "-m", "jupyter", "nbconvert",
            "--to", "notebook",
            "--execute",
            "--ExecutePreprocessor.timeout=" + str(timeout_minutes * 60),
            "--ExecutePreprocessor.kernel_name=python3",
            "--output", executed_path,
            notebook_path,
        ]
        
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout_minutes * 60 + 60,  # Extra buffer for nbconvert overhead
            cwd=working_dir or os.path.dirname(notebook_path),
        )
        
        elapsed = time.time() - start_time
        
        if result.returncode == 0:
            return {
                "status": "SUCCESS",
                "elapsed_seconds": elapsed,
                "elapsed_str": str(timedelta(seconds=int(elapsed))),
                "executed_path": executed_path if SAVE_EXECUTED_NOTEBOOKS else None,
                "stdout": result.stdout[-500:] if result.stdout else "",
                "stderr": "",
            }
        else:
            # Extract error from stderr
            error_msg = result.stderr[-1000:] if result.stderr else "Unknown error"
            return {
                "status": "FAILED",
                "elapsed_seconds": elapsed,
                "elapsed_str": str(timedelta(seconds=int(elapsed))),
                "error": error_msg,
                "stdout": result.stdout[-500:] if result.stdout else "",
                "stderr": result.stderr[-500:] if result.stderr else "",
            }
    
    except subprocess.TimeoutExpired:
        elapsed = time.time() - start_time
        return {
            "status": "TIMEOUT",
            "elapsed_seconds": elapsed,
            "elapsed_str": str(timedelta(seconds=int(elapsed))),
            "error": f"Notebook exceeded {timeout_minutes} minute timeout",
        }
    except Exception as e:
        elapsed = time.time() - start_time
        return {
            "status": "ERROR",
            "elapsed_seconds": elapsed,
            "elapsed_str": str(timedelta(seconds=int(elapsed))),
            "error": str(e),
        }


def validate_outputs(stage, output_dir):
    """Check that expected output files were created by a stage.

    Paths in stage["outputs"] may be:
      - absolute  -> used verbatim
      - relative  -> resolved against output_dir (the orchestrator's OUTPUT_DIR)

    The model-training stages (04, 05) write outside outputs_updated/ — those
    entries are configured as absolute paths in PIPELINE.
    """
    results = []
    for output_file in stage["outputs"]:
        full_path = output_file if os.path.isabs(output_file) else os.path.join(output_dir, output_file)
        # Display name: keep absolute paths short by showing only the tail
        if os.path.isabs(output_file):
            display = os.path.relpath(output_file, os.path.dirname(output_dir))
        else:
            display = output_file
        if os.path.exists(full_path):
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            results.append({"file": display, "status": "✓", "size_mb": size_mb})
        else:
            results.append({"file": display, "status": "⚠ MISSING", "size_mb": 0})
    return results


print("Runner engine loaded.")
print("  run_notebook()     — executes a notebook via nbconvert")
print("  validate_outputs() — checks expected output files exist")


Runner engine loaded.
  run_notebook()     — executes a notebook via nbconvert
  validate_outputs() — checks expected output files exist


## 3 — Execute Full Pipeline

This cell runs all 6 notebooks sequentially. Each notebook:
1. Executes in its own isolated kernel
2. Has its outputs validated after completion
3. Logs timing and status

**Estimated total runtime:** 15–45 minutes depending on hardware.


In [ ]:
print(f"{'=' * 75}")
print("PIPELINE EXECUTION STARTED")
print(f"  Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

pipeline_start = time.time()
pipeline_results = []
pipeline_failed = False

for i, stage in enumerate(PIPELINE):
    print(f"\n{'─' * 70}")
    print(f"  STAGE [{stage['id']}] {stage['name']}")
    print(f"  {stage['description']}")
    print(f"{'─' * 70}")
    
    if pipeline_failed and STOP_ON_FAILURE:
        print(f"  ⊘ SKIPPED (previous stage failed)")
        pipeline_results.append({
            "stage": stage["id"],
            "name": stage["name"],
            "status": "SKIPPED",
            "elapsed_str": "—",
            "elapsed_seconds": 0,
        })
        continue
    
    # Execute notebook
    nb_path = os.path.join(NOTEBOOK_DIR, stage["notebook"])
    result = run_notebook(nb_path, timeout_minutes=TIMEOUT_MINUTES, working_dir=NOTEBOOK_DIR)
    
    # Status indicator
    status_icon = {"SUCCESS": "✓", "FAILED": "✗", "TIMEOUT": "⏱", "ERROR": "⚠"}.get(result["status"], "?")
    print(f"\n  {status_icon} Status: {result['status']} ({result['elapsed_str']})")
    
    if result["status"] != "SUCCESS":
        print(f"  Error: {result.get('error', 'Unknown')[:300]}")
        if result.get("stderr"):
            print(f"  Stderr: {result['stderr'][:300]}")
        pipeline_failed = True
    
    # Validate outputs
    if result["status"] == "SUCCESS":
        validations = validate_outputs(stage, OUTPUT_DIR)
        print(f"\n  Output Validation:")
        all_valid = True
        for v in validations:
            print(f"    {v['status']} {v['file']:<50s} {v['size_mb']:>8.2f} MB")
            if v["status"] != "✓":
                all_valid = False
        
        if not all_valid:
            print(f"\n  ⚠ WARNING: Some expected outputs are missing.")
            print(f"    Pipeline will continue but downstream modules may fail.")
    
    pipeline_results.append({
        "stage": stage["id"],
        "name": stage["name"],
        "notebook": stage["notebook"],
        "status": result["status"],
        "elapsed_str": result["elapsed_str"],
        "elapsed_seconds": result["elapsed_seconds"],
    })

pipeline_elapsed = time.time() - pipeline_start
print(f"\n{'=' * 70}")
print(f"PIPELINE EXECUTION COMPLETE")
print(f"  Total time: {str(timedelta(seconds=int(pipeline_elapsed)))}")
print(f"  End time:   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'=' * 70}")


PIPELINE EXECUTION STARTED
  Start time: 2026-05-15 12:26:55

──────────────────────────────────────────────────────────────────────
  STAGE [00] Data Generator
  Generate synthetic transactions with 10 AML typologies
──────────────────────────────────────────────────────────────────────
  Executing: aml_generator_complete_pipeline.ipynb
  Working dir: c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts

  ✓ Status: SUCCESS (0:00:45)

  Output Validation:
    ✓ transactions_generated_typology.parquet               45.06 MB

──────────────────────────────────────────────────────────────────────
  STAGE [01] Typology Detector
  Graph-based detection of AML patterns, assign is_aml labels
──────────────────────────────────────────────────────────────────────
  Executing: 01__aml_typology_detector.ipynb
  Working dir: c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts

  ✓ Status: SUCCESS (0:10:35)

  Output 

## 4 — Execution Summary


In [ ]:
print("\n" + "=" * 70)
print("PIPELINE SUMMARY REPORT")
print("=" * 70)

print(f"\n  {'Stage':<6s} {'Module':<35s} {'Status':<10s} {'Time':>10s}")
print(f"  {'─' * 65}")

total_success = 0
total_failed = 0
total_skipped = 0

for r in pipeline_results:
    icon = {"SUCCESS":"✓","FAILED":"✗","TIMEOUT":"⏱","SKIPPED":"⊘","ERROR":"⚠"}.get(r["status"],"?")
    print(f"  [{r['stage']}]  {r['name']:<35s} {icon} {r['status']:<8s} {r['elapsed_str']:>10s}")
    
    if r["status"] == "SUCCESS": total_success += 1
    elif r["status"] == "SKIPPED": total_skipped += 1
    else: total_failed += 1

print(f"\n  Total: {total_success} succeeded | {total_failed} failed | {total_skipped} skipped")
print(f"  Pipeline time: {str(timedelta(seconds=int(pipeline_elapsed)))}")

# Overall status
if total_failed == 0 and total_skipped == 0:
    print(f"\n  ✓ PIPELINE COMPLETED SUCCESSFULLY")
elif total_failed > 0:
    print(f"\n  ✗ PIPELINE FAILED — check error logs above")
    failed_stages = [r for r in pipeline_results if r["status"] in ("FAILED", "ERROR", "TIMEOUT")]
    for r in failed_stages:
        print(f"    → Stage [{r['stage']}] {r['name']}: {r['status']}")

# ═══ Output file inventory ═══
print(f"\n{'─' * 70}")
print(f"OUTPUT FILE INVENTORY")
print(f"{'─' * 70}")

total_size = 0
file_count = 0
for root, dirs, files in os.walk(OUTPUT_DIR):
    # Skip executed_notebooks directory for cleaner output
    if "executed_notebooks" in root:
        continue
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        rel_path = os.path.relpath(fpath, OUTPUT_DIR)
        print(f"  {rel_path:<60s} {size_mb:>8.2f} MB")
        total_size += size_mb
        file_count += 1

print(f"\n  Total: {file_count} files, {total_size:.2f} MB")
print(f"  Location: {OUTPUT_DIR}")



PIPELINE SUMMARY REPORT

  Stage  Module                              Status           Time
  ─────────────────────────────────────────────────────────────────
  [00]  Data Generator                      ✓ SUCCESS     0:00:45
  [01]  Typology Detector                   ✓ SUCCESS     0:10:35
  [02]  Rules Engine                        ✓ SUCCESS     0:03:32
  [03]  Feature Engineering                 ✓ SUCCESS     0:07:01
  [04]  Phase 1: AML Detection              ✓ SUCCESS     0:13:02
  [05]  Phase 2: Typology Classifier        ✓ SUCCESS     0:37:49

  Total: 6 succeeded | 0 failed | 0 skipped
  Pipeline time: 1:12:46

  ✓ PIPELINE COMPLETED SUCCESSFULLY

──────────────────────────────────────────────────────────────────────
OUTPUT FILE INVENTORY
──────────────────────────────────────────────────────────────────────
  accounts.csv                                                     0.46 MB
  alert_transactions.parquet                                      11.95 MB
  config.json        

## 5 — Quick Output Validation

Loads key output files and prints summary statistics to confirm the pipeline produced valid results.


In [ ]:
import pandas as pd

print("\n" + "=" * 70)
print("QUICK VALIDATION")
print("=" * 70)

validation_checks = []

# Check 1: Generator output
gen_file = os.path.join(OUTPUT_DIR, "transactions_generated_typology.parquet")
if os.path.exists(gen_file):
    df_gen = pd.read_parquet(gen_file)
    aml_count = (df_gen.get("is_aml", pd.Series()) == 1).sum()
    total = len(df_gen)
    fraud_rate = aml_count / total * 100 if total > 0 else 0
    print(f"\n  [00] Generator: {total:,} transactions, {aml_count:,} AML ({fraud_rate:.1f}%)")
    validation_checks.append(("Generator", total > 300000 and 15 < fraud_rate < 30))
    del df_gen
else:
    print(f"\n  [00] Generator: ⚠ Output not found")
    validation_checks.append(("Generator", False))

# Check 2: Detector output
det_file = os.path.join(OUTPUT_DIR, "stg_transactions_flagged.parquet")
if os.path.exists(det_file):
    df_det = pd.read_parquet(det_file)
    flagged = (df_det.get("is_aml", pd.Series()) == 1).sum()
    typs = df_det.get("aml_typology", pd.Series()).nunique()
    multi = df_det.get("aml_typology", pd.Series()).astype(str).str.contains(";", na=False).sum()
    print(f"  [01] Detector:  {flagged:,} flagged, {typs} typologies, {multi} multi-label (should be 0)")
    validation_checks.append(("Detector", flagged > 50000 and multi == 0))
    del df_det
else:
    print(f"  [01] Detector:  ⚠ Output not found")
    validation_checks.append(("Detector", False))

# Check 3: Rules output
rules_file = os.path.join(OUTPUT_DIR, "stg_transactions_rules.parquet")
if os.path.exists(rules_file):
    df_rules = pd.read_parquet(rules_file)
    rule_cols = [c for c in df_rules.columns if c.startswith("rule_") and c not in {"rule_score","rules_triggered","rules_triggered_count"}]
    trigger_rate = (df_rules[rule_cols].sum(axis=1) > 0).mean() * 100 if rule_cols else 0
    print(f"  [02] Rules:     {len(rule_cols)} rules, {trigger_rate:.1f}% trigger rate (target: 50-60%)")
    validation_checks.append(("Rules", 40 < trigger_rate < 75))
    del df_rules
else:
    print(f"  [02] Rules:     ⚠ Output not found")
    validation_checks.append(("Rules", False))

# Check 4: Features output
feat_file = os.path.join(OUTPUT_DIR, "stg_transactions_features.parquet")
if os.path.exists(feat_file):
    df_feat = pd.read_parquet(feat_file)
    n_features = len(df_feat.columns)
    print(f"  [03] Features:  {len(df_feat):,} rows × {n_features} columns")
    validation_checks.append(("Features", n_features > 150))
    del df_feat
else:
    print(f"  [03] Features:  ⚠ Output not found")
    validation_checks.append(("Features", False))

# Check 5: Phase 1 model
model_file = os.path.join(OUTPUT_DIR, "ml_outputs", "model_metadata.json")
if os.path.exists(model_file):
    with open(model_file) as f:
        meta = json.load(f)
    print(f"  [04] Phase 1:   AUC={meta.get('auc_roc','?'):.4f}, F1={meta.get('f1_score','?'):.4f}, "
          f"Threshold={meta.get('optimal_threshold','?')}, Features={meta.get('n_features','?')}")
    auc = meta.get("auc_roc", 0)
    validation_checks.append(("Phase 1", auc > 0.90))
else:
    print(f"  [04] Phase 1:   ⚠ Model metadata not found")
    validation_checks.append(("Phase 1", False))

# Check 6: Phase 2 model
p2_meta_file = os.path.join(OUTPUT_DIR, "phase2_outputs", "phase2_metadata.json")
if os.path.exists(p2_meta_file):
    with open(p2_meta_file) as f:
        p2_meta = json.load(f)
    print(f"  [05] Phase 2:   Accuracy={p2_meta.get('phase2_accuracy','?'):.4f}, "
          f"Classes={p2_meta.get('n_classes','?')}, Features={p2_meta.get('n_features','?')}")
    p2_acc = p2_meta.get("phase2_accuracy", 0)
    validation_checks.append(("Phase 2", p2_acc > 0.70))
else:
    print(f"  [05] Phase 2:   ⚠ Model metadata not found")
    validation_checks.append(("Phase 2", False))

# Combined output check
combined_file = os.path.join(OUTPUT_DIR, "phase2_outputs", "combined_aml_output.parquet")
if os.path.exists(combined_file):
    df_combined = pd.read_parquet(combined_file)
    aml_alerts = (df_combined.get("predicted_typology", pd.Series()) != "None").sum()
    multi_label = (df_combined.get("num_typologies_matched", pd.Series(0)) >= 2).sum()
    print(f"\n  Combined Output: {len(df_combined):,} rows, {aml_alerts:,} AML alerts, {multi_label:,} multi-label")
    
    # Priority breakdown
    if "investigation_priority" in df_combined.columns:
        print(f"  Priority: ", end="")
        for pri in ["Critical", "High", "Medium", "Low"]:
            cnt = (df_combined["investigation_priority"] == pri).sum()
            print(f"{pri}={cnt:,} ", end="")
        print()
    del df_combined
else:
    print(f"\n  Combined Output: ⚠ Not found")

# Final verdict
print(f"\n{'─' * 70}")
print(f"VALIDATION SUMMARY")
print(f"{'─' * 70}")
all_passed = True
for name, passed in validation_checks:
    icon = "✓" if passed else "✗"
    print(f"  {icon} {name}")
    if not passed: all_passed = False

if all_passed:
    print(f"\n  ✓ ALL VALIDATIONS PASSED — Pipeline output is ready for deployment")
else:
    print(f"\n  ⚠ SOME VALIDATIONS FAILED — Review output above for details")



QUICK VALIDATION

  [00] Generator: 386,570 transactions, 85,555 AML (22.1%)
  [01] Detector:  72,854 flagged, 11 typologies, 0 multi-label (should be 0)
  [02] Rules:     126 rules, 84.9% trigger rate (target: 50-60%)
  [03] Features:  386,570 rows × 318 columns
  [04] Phase 1:   ⚠ Model metadata not found
  [05] Phase 2:   ⚠ Model metadata not found

  Combined Output: ⚠ Not found

──────────────────────────────────────────────────────────────────────
VALIDATION SUMMARY
──────────────────────────────────────────────────────────────────────
  ✓ Generator
  ✓ Detector
  ✗ Rules
  ✓ Features
  ✗ Phase 1
  ✗ Phase 2

  ⚠ SOME VALIDATIONS FAILED — Review output above for details


## 6 — Run Individual Modules (Optional)

Use this cell to re-run a single module without executing the full pipeline. Useful for debugging or re-running a specific stage after fixing an issue.

**Change `MODULE_TO_RUN`** to the module ID you want to execute (00–05).


In [ ]:
# # ═══ Change this to run a specific module ═══
# MODULE_TO_RUN = "01"  # Options: "00", "01", "02", "03", "04", "05"

# # Find the module
# target = next((s for s in PIPELINE if s["id"] == MODULE_TO_RUN), None)
# if not target:
#     print(f"Module {MODULE_TO_RUN} not found. Valid IDs: {[s['id'] for s in PIPELINE]}")
# else:
#     print(f"Running single module: [{target['id']}] {target['name']}")
#     print(f"  {target['description']}")
#     print(f"  Notebook: {target['notebook']}")
#     print()
    
#     nb_path = os.path.join(NOTEBOOK_DIR, target["notebook"])
#     result = run_notebook(nb_path, timeout_minutes=TIMEOUT_MINUTES, working_dir=NOTEBOOK_DIR)
    
#     icon = {"SUCCESS":"✓","FAILED":"✗","TIMEOUT":"⏱","ERROR":"⚠"}.get(result["status"],"?")
#     print(f"\n  {icon} {result['status']} ({result['elapsed_str']})")
    
#     if result["status"] == "SUCCESS":
#         validations = validate_outputs(target, OUTPUT_DIR)
#         print(f"\n  Outputs:")
#         for v in validations:
#             print(f"    {v['status']} {v['file']:<50s} {v['size_mb']:>8.2f} MB")
#     else:
#         print(f"  Error: {result.get('error', 'Unknown')[:500]}")
#         if result.get("stderr"):
#             print(f"\n  Stderr (last 500 chars):")
#             print(f"  {result['stderr'][:500]}")


## 7 — Run Pipeline from a Specific Stage (Optional)

If a module failed and you've fixed it, use this to resume from that stage instead of re-running the entire pipeline. All subsequent modules will also be re-run.


In [ ]:
# # ═══ Change this to the stage to START from ═══
# START_FROM = "02"  # Will run 02, 03, 04, 05

# print(f"Running pipeline from stage [{START_FROM}] onwards...")
# print()

# start_idx = next((i for i, s in enumerate(PIPELINE) if s["id"] == START_FROM), None)
# if start_idx is None:
#     print(f"Stage {START_FROM} not found. Valid IDs: {[s['id'] for s in PIPELINE]}")
# else:
#     stages_to_run = PIPELINE[start_idx:]
#     print(f"  Stages to execute: {[s['id'] + ' ' + s['name'] for s in stages_to_run]}")
#     print()
    
#     partial_results = []
#     failed = False
#     partial_start = time.time()
    
#     for stage in stages_to_run:
#         print(f"{'─' * 50}")
#         print(f"  [{stage['id']}] {stage['name']}")
        
#         if failed and STOP_ON_FAILURE:
#             print(f"  ⊘ SKIPPED")
#             partial_results.append({"stage": stage["id"], "name": stage["name"], "status": "SKIPPED", "elapsed_str": "—"})
#             continue
        
#         nb_path = os.path.join(NOTEBOOK_DIR, stage["notebook"])
#         result = run_notebook(nb_path, timeout_minutes=TIMEOUT_MINUTES, working_dir=NOTEBOOK_DIR)
        
#         icon = {"SUCCESS":"✓","FAILED":"✗","TIMEOUT":"⏱","ERROR":"⚠"}.get(result["status"],"?")
#         print(f"  {icon} {result['status']} ({result['elapsed_str']})")
        
#         if result["status"] != "SUCCESS":
#             print(f"  Error: {result.get('error', 'Unknown')[:300]}")
#             failed = True
#         else:
#             validations = validate_outputs(stage, OUTPUT_DIR)
#             for v in validations:
#                 print(f"    {v['status']} {v['file']}")
        
#         partial_results.append({"stage": stage["id"], "name": stage["name"], "status": result["status"], "elapsed_str": result["elapsed_str"]})
    
#     partial_elapsed = time.time() - partial_start
#     print(f"\n{'─' * 50}")
#     print(f"  Partial pipeline complete: {str(timedelta(seconds=int(partial_elapsed)))}")
#     for r in partial_results:
#         icon = {"SUCCESS":"✓","FAILED":"✗","TIMEOUT":"⏱","SKIPPED":"⊘"}.get(r["status"],"?")
#         print(f"    {icon} [{r['stage']}] {r['name']}: {r['status']} ({r['elapsed_str']})")


## 8 — Save Execution Log


In [ ]:
# Save pipeline execution log as JSON
log = {
    "pipeline_name": "SmartSentry AML",
    "execution_date": datetime.now().isoformat(),
    "total_elapsed_seconds": pipeline_elapsed,
    "total_elapsed_str": str(timedelta(seconds=int(pipeline_elapsed))),
    "notebook_dir": NOTEBOOK_DIR,
    "output_dir": OUTPUT_DIR,
    "stages": pipeline_results,
    "overall_status": "SUCCESS" if all(r["status"] == "SUCCESS" for r in pipeline_results) else "FAILED",
}

log_path = os.path.join(OUTPUT_DIR, "pipeline_execution_log.json")
with open(log_path, "w") as f:
    json.dump(log, f, indent=2, default=str)

print(f"Execution log saved: {log_path}")
print(f"\n{'=' * 70}")
print(f"ORCHESTRATOR COMPLETE")
print(f"{'=' * 70}")


Execution log saved: c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\pipeline_execution_log.json

ORCHESTRATOR COMPLETE


## 9 — Persistent Run History Log

In [ ]:
# ════════════════════════════════════════════════════════════════
# PERSISTENT RUN LOG
# ════════════════════════════════════════════════════════════════
# Every pipeline run (training or inference) appends one JSON line to
# outputs_updated/run_history.jsonl. This file is the single source of
# truth for run history and grows monotonically — never overwritten.
#
# Each entry contains:
#   - run_id (timestamped, unique)
#   - timestamp (ISO 8601, local time)
#   - mode ("train" or "predict")
#   - pipeline_duration_seconds
#   - stage statuses (pass/fail per notebook)
#   - phase1 metrics  (AUC, F1, recall, threshold, ...)
#   - phase2 metrics  (primary/multi-label accuracy, threshold)
#   - row counts (input + predicted-AML if available)
#   - git_hash if .git is present
#
# Tail with:
#   python -c "import json; [print(json.dumps(json.loads(l), indent=2)) for l in open('outputs_updated/run_history.jsonl')]"
#
# Or load as a DataFrame:
#   pd.read_json("outputs_updated/run_history.jsonl", lines=True)
# ════════════════════════════════════════════════════════════════

import json as _json
import os as _os
import subprocess as _subprocess
from datetime import datetime as _datetime

def _safe_json_read(path):
    """Read a JSON file if it exists; return {} otherwise."""
    if not _os.path.exists(path):
        return {}
    try:
        with open(path) as _f:
            return _json.load(_f)
    except Exception:
        return {}

def _safe_parquet_row_count(path):
    """Count rows in a parquet file without loading it fully."""
    if not _os.path.exists(path):
        return None
    try:
        import pyarrow.parquet as _pq
        return int(_pq.ParquetFile(path).metadata.num_rows)
    except Exception:
        try:
            import pandas as _pd
            return int(len(_pd.read_parquet(path)))
        except Exception:
            return None

def _git_hash():
    try:
        out = _subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True, timeout=2)
        return out.stdout.strip() if out.returncode == 0 else None
    except Exception:
        return None

def write_run_log(mode, pipeline_results, pipeline_elapsed_seconds,
                  output_dir, phase1_dir, phase2_dir):
    """Append one entry to outputs_updated/run_history.jsonl."""
    now = _datetime.now()
    run_id = now.strftime("%Y%m%d_%H%M%S")

    # Phase 1 metrics — from model_parameters_full.json saved by 04
    p1_path = _os.path.join(phase1_dir, "model_parameters_full.json")
    p1_full = _safe_json_read(p1_path)
    p1_metrics = {}
    if "phase1" in p1_full:
        p1 = p1_full["phase1"]
        p1_metrics = {
            "best_config":     p1.get("best_config"),
            "threshold":       p1.get("threshold"),
            "auc_roc":         p1.get("auc_roc"),
            "f1_score":        p1.get("f1_score"),
            "precision":       p1.get("precision"),
            "recall":          p1.get("recall"),
            "tp":              p1.get("tp"),
            "fp":              p1.get("fp"),
            "fn":              p1.get("fn"),
            "tn":              p1.get("tn"),
            "best_iteration":  p1.get("best_iteration"),
            "n_features":      p1.get("n_features"),
            "n_train":         p1.get("n_train"),
            "n_test":          p1.get("n_test"),
            "imbalance_ratio": p1.get("imbalance_ratio"),
        }

    # Phase 2 metrics — from model_parameters_full.json saved by 05
    p2_path = _os.path.join(phase2_dir, "model_parameters_full.json")
    p2_full = _safe_json_read(p2_path)
    p2_metrics = {}
    if "phase2" in p2_full:
        p2 = p2_full["phase2"]
        p2_metrics = {
            "best_config":           p2.get("best_config"),
            "accuracy_primary":      p2.get("accuracy_primary"),
            "accuracy_multi_label":  p2.get("accuracy_multi_label"),
            "multi_label_threshold": p2.get("multi_label_threshold"),
            "macro_f1":              p2.get("macro_f1"),
            "weighted_f1":           p2.get("weighted_f1"),
            "best_iteration":        p2.get("best_iteration"),
            "n_classes":             p2.get("n_classes"),
            "n_features":            p2.get("n_features"),
            "n_train":                p2.get("n_train"),
            "n_test":                 p2.get("n_test"),
        }

    # Row counts — best-effort, parquets may not exist
    row_counts = {
        "rules_engine":        _safe_parquet_row_count(_os.path.join(output_dir, "stg_transactions_rules.parquet")),
        "feature_engineering": _safe_parquet_row_count(_os.path.join(output_dir, "stg_transactions_features.parquet")),
        "phase1_scored":       _safe_parquet_row_count(_os.path.join(phase1_dir, "df_ml_phase_1.parquet")),
        "phase2_predictions":  _safe_parquet_row_count(_os.path.join(phase2_dir, "predictions_output.parquet")),
    }
    # If in predict mode, the last parquet has the final predicted-AML count
    n_aml_predicted = None
    try:
        pred_path = _os.path.join(phase2_dir, "predictions_output.parquet")
        if _os.path.exists(pred_path):
            import pandas as _pd
            _pred = _pd.read_parquet(pred_path, columns=["predicted_typology"])
            n_aml_predicted = int((_pred["predicted_typology"].fillna("None") != "None").sum())
    except Exception:
        pass

    entry = {
        "run_id":      run_id,
        "timestamp":   now.isoformat(timespec="seconds"),
        "mode":        mode,
        "duration_seconds": round(pipeline_elapsed_seconds, 2),
        "duration_str":     str(_datetime.fromtimestamp(pipeline_elapsed_seconds, tz=None).strftime("%H:%M:%S")) if pipeline_elapsed_seconds > 0 else "—",
        "git_hash":    _git_hash(),
        "stages": [
            {"id": r["stage"], "name": r["name"], "status": r["status"], "duration": r.get("elapsed_str", "—")}
            for r in pipeline_results
        ],
        "stages_succeeded": sum(1 for r in pipeline_results if r["status"] == "SUCCESS"),
        "stages_total":     len(pipeline_results),
        "phase1_metrics":   p1_metrics,
        "phase2_metrics":   p2_metrics,
        "row_counts":       row_counts,
        "n_aml_predicted":  n_aml_predicted,
    }

    log_path = _os.path.join(output_dir, "run_history.jsonl")
    _os.makedirs(output_dir, exist_ok=True)
    with open(log_path, "a") as _f:
        _f.write(_json.dumps(entry, default=str) + "\n")

    # Pretty print summary
    print()
    print("=" * 70)
    print(f"RUN LOG — entry written to {log_path}")
    print("=" * 70)
    print(f"  Run ID:    {run_id}")
    print(f"  Mode:      {mode.upper()}")
    print(f"  Duration:  {entry['duration_seconds']}s")
    print(f"  Stages:    {entry['stages_succeeded']}/{entry['stages_total']} succeeded")
    if p1_metrics:
        print(f"  Phase 1:   AUC={p1_metrics.get('auc_roc'):.4f}  Recall={p1_metrics.get('recall'):.4f}  F1={p1_metrics.get('f1_score'):.4f}" if p1_metrics.get('auc_roc') else "  Phase 1:   (metrics unavailable)")
    if p2_metrics:
        ap = p2_metrics.get('accuracy_primary')
        am = p2_metrics.get('accuracy_multi_label')
        if ap and am:
            print(f"  Phase 2:   primary={ap*100:.2f}%  multi-label={am*100:.2f}%")
    if n_aml_predicted is not None:
        print(f"  Predicted AML rows: {n_aml_predicted:,}")
    print(f"  Total history entries: {sum(1 for _ in open(log_path))}")
    print("=" * 70)
    return log_path

# Call the writer — pipeline_results / pipeline_elapsed must be in scope.
# Determine mode automatically: training orchestrator has PIPELINE_MODE,
# inference orchestrator runs only in predict mode.
_log_mode = "train" if "PIPELINE_MODE" in dir() and PIPELINE_MODE == "train" else "predict"
_log_phase1_dir = _os.environ.get("AML_PHASE1_DIR",
    _os.path.join(_os.path.dirname(_os.getcwd()), "python_scripts", "ml_outputs"))
_log_phase2_dir = _os.environ.get("AML_PHASE2_DIR",
    _os.path.join(_os.path.dirname(_os.getcwd()), "python_scripts", "phase2_outputs"))

write_run_log(_log_mode, pipeline_results, pipeline_elapsed,
              OUTPUT_DIR, _log_phase1_dir, _log_phase2_dir)



RUN LOG — entry written to c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\run_history.jsonl
  Run ID:    20260515_133944
  Mode:      TRAIN
  Duration:  4366.61s
  Stages:    6/6 succeeded
  Phase 1:   AUC=0.9420  Recall=0.8562  F1=0.7476
  Phase 2:   primary=75.73%  multi-label=81.78%
  Total history entries: 3


'c:\\Users\\VISHNUPRIYA\\OneDrive\\Desktop\\Freelancing\\AIGEN\\smartsentry_aml_model\\outputs_updated\\run_history.jsonl'

## 10 — Dashboard View *(monitoring summary)*

In [ ]:
# ════════════════════════════════════════════════════════════════
# DASHBOARD VIEW — flatten run history into a monitoring table
# ════════════════════════════════════════════════════════════════
# Reads outputs_updated/run_history.jsonl (always present after the
# previous cell wrote a new entry). Builds / rebuilds a flat
# DataFrame with one row per run and saves it as
# outputs_updated/run_dashboard.csv — overwritten each time but
# always reflecting the COMPLETE history.
#
# Past runs are preserved by run_history.jsonl (append-only).
# The CSV is a derived view that's easy to open in Excel.
# ════════════════════════════════════════════════════════════════

import json as _json
import os as _os
import pandas as _pd


def update_run_dashboard(output_dir, recent_n=10):
    """Build / refresh the monitoring dashboard from run_history.jsonl.

    - If run_history.jsonl does not exist: returns an empty DataFrame.
    - If run_dashboard.csv does not exist: creates it.
    - If run_dashboard.csv already exists: overwrites with the full
      history (jsonl is the source of truth — CSV is a derived view).

    Parameters
    ----------
    output_dir : str
        Directory containing run_history.jsonl. Dashboard CSV is saved
        next to it.
    recent_n : int, default 10
        How many most-recent runs to print in the inline summary.

    Returns
    -------
    pandas.DataFrame
        One row per run, flattened. Empty if no history exists.
    """
    log_path  = _os.path.join(output_dir, "run_history.jsonl")
    dash_path = _os.path.join(output_dir, "run_dashboard.csv")

    # ─── Read every line from the JSONL log ────────────────────────
    if not _os.path.exists(log_path):
        print(f"⚠  No run history file at {log_path} — nothing to dashboard yet.")
        return _pd.DataFrame()

    rows = []
    with open(log_path) as _f:
        for line_no, raw in enumerate(_f, start=1):
            raw = raw.strip()
            if not raw:
                continue
            try:
                rows.append(_json.loads(raw))
            except _json.JSONDecodeError as exc:
                print(f"⚠  Skipping malformed log line {line_no}: {exc}")
                continue

    if not rows:
        print(f"⚠  Log file at {log_path} is empty.")
        return _pd.DataFrame()

    # ─── Flatten each entry into a single row ─────────────────────
    flat = []
    for r in rows:
        p1 = r.get("phase1_metrics") or {}
        p2 = r.get("phase2_metrics") or {}
        rc = r.get("row_counts") or {}
        flat.append({
            "run_id":            r.get("run_id"),
            "timestamp":         r.get("timestamp"),
            "mode":              r.get("mode"),
            "duration_seconds":  r.get("duration_seconds"),
            "stages_succeeded":  r.get("stages_succeeded"),
            "stages_total":      r.get("stages_total"),
            "git_hash":          r.get("git_hash"),
            # Phase 1
            "p1_best_config":    p1.get("best_config"),
            "p1_threshold":      p1.get("threshold"),
            "p1_auc_roc":        p1.get("auc_roc"),
            "p1_f1_score":       p1.get("f1_score"),
            "p1_precision":      p1.get("precision"),
            "p1_recall":         p1.get("recall"),
            "p1_tp":             p1.get("tp"),
            "p1_fp":             p1.get("fp"),
            "p1_fn":             p1.get("fn"),
            "p1_tn":             p1.get("tn"),
            "p1_n_features":     p1.get("n_features"),
            "p1_n_train":        p1.get("n_train"),
            "p1_n_test":         p1.get("n_test"),
            "p1_imbalance":      p1.get("imbalance_ratio"),
            # Phase 2
            "p2_best_config":         p2.get("best_config"),
            "p2_accuracy_primary":    p2.get("accuracy_primary"),
            "p2_accuracy_multilabel": p2.get("accuracy_multi_label"),
            "p2_multilabel_thresh":   p2.get("multi_label_threshold"),
            "p2_macro_f1":            p2.get("macro_f1"),
            "p2_weighted_f1":         p2.get("weighted_f1"),
            "p2_n_classes":           p2.get("n_classes"),
            "p2_n_train":             p2.get("n_train"),
            "p2_n_test":              p2.get("n_test"),
            # Row counts + final prediction count
            "n_rules":                 rc.get("rules_engine"),
            "n_features_eng":          rc.get("feature_engineering"),
            "n_phase1_scored":         rc.get("phase1_scored"),
            "n_phase2_predictions":    rc.get("phase2_predictions"),
            "n_aml_predicted":         r.get("n_aml_predicted"),
        })

    df_dash = _pd.DataFrame(flat)

    # Sort newest-first for display; CSV stays chronological
    df_dash_csv = df_dash.sort_values("timestamp", ascending=True).reset_index(drop=True)

    # ─── Write CSV (create or overwrite — full history every time) ─
    new_file = not _os.path.exists(dash_path)
    df_dash_csv.to_csv(dash_path, index=False)
    action = "Created" if new_file else "Refreshed"
    print(f"{action} dashboard CSV:")
    print(f"  Path:     {dash_path}")
    print(f"  Size:     {_os.path.getsize(dash_path)/1024:.1f} KB")
    print(f"  Runs:     {len(df_dash_csv)}")
    print(f"  Columns:  {len(df_dash_csv.columns)}")

    # ─── Recent runs summary (printed inline) ──────────────────────
    df_recent = df_dash.sort_values("timestamp", ascending=False).head(recent_n).copy()
    cols_to_show = [
        "run_id", "mode", "duration_seconds",
        "p1_auc_roc", "p1_recall", "p1_f1_score",
        "p2_accuracy_primary", "p2_accuracy_multilabel",
        "n_aml_predicted",
    ]
    df_show = df_recent[cols_to_show].copy()

    # Round numeric columns for readability
    for c in ["p1_auc_roc", "p1_recall", "p1_f1_score",
              "p2_accuracy_primary", "p2_accuracy_multilabel"]:
        if c in df_show.columns:
            df_show[c] = df_show[c].apply(lambda v: f"{v:.4f}" if _pd.notna(v) else "—")
    df_show["duration_seconds"] = df_show["duration_seconds"].apply(
        lambda v: f"{v:.1f}s" if _pd.notna(v) else "—")
    df_show["n_aml_predicted"] = df_show["n_aml_predicted"].apply(
        lambda v: f"{int(v):,}" if _pd.notna(v) else "—")

    print()
    print("=" * 95)
    print(f"DASHBOARD — last {min(recent_n, len(df_dash))} run(s) (newest first)")
    print("=" * 95)
    print(df_show.to_string(index=False))
    print("=" * 95)

    # ─── Quick aggregate stats (mean / std over all training runs) ─
    train_mask = df_dash["mode"] == "train"
    if train_mask.sum() >= 2:
        train_df = df_dash[train_mask]
        print()
        print("Training-run aggregates (mean ± std across all training runs):")
        for col, label in [
            ("p1_auc_roc",            "Phase 1 AUC-ROC"),
            ("p1_recall",             "Phase 1 Recall"),
            ("p1_f1_score",           "Phase 1 F1"),
            ("p2_accuracy_primary",   "Phase 2 primary acc"),
            ("p2_accuracy_multilabel","Phase 2 multi-label acc"),
        ]:
            vals = _pd.to_numeric(train_df[col], errors="coerce").dropna()
            if len(vals) >= 2:
                print(f"  {label:<28s} {vals.mean():.4f} ± {vals.std():.4f}  (n={len(vals)})")

    return df_dash


# ─── Run it ────────────────────────────────────────────────────────
_dashboard_df = update_run_dashboard(OUTPUT_DIR, recent_n=10)


Refreshed dashboard CSV:
  Path:     c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\run_dashboard.csv
  Size:     1.4 KB
  Runs:     3
  Columns:  35

DASHBOARD — last 3 run(s) (newest first)
         run_id  mode duration_seconds p1_auc_roc p1_recall p1_f1_score p2_accuracy_primary p2_accuracy_multilabel n_aml_predicted
20260515_133944 train          4366.6s     0.9420    0.8562      0.7476              0.7573                 0.8178               —
20260515_121329 train          1996.7s     0.9420    0.8562      0.7476              0.7573                 0.8178               —
20260515_112849 train          1870.4s     0.9420    0.8562      0.7476              0.7573                 0.8178               —

Training-run aggregates (mean ± std across all training runs):
  Phase 1 AUC-ROC              0.9420 ± 0.0000  (n=3)
  Phase 1 Recall               0.8562 ± 0.0000  (n=3)
  Phase 1 F1                   0.7476 ± 0.0000  (n=3)
  Phase 2 p